In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-16'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 75.0]}}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.1150, -0.4421, -0.4975,  0.3258, -0.9106,  0.0355, -0.0733,  0.4647,
          0.2896,  0.1431, -1.0629,  0.1772]], device='cuda:0')
Scaled actions :  tensor([[ 0.1150, -0.4421, -0.4975,  0.3258, -0.9106,  0.0355, -0.0733,  0.4647,
          0.2896,  0.1431, -1.0629,  0.1772]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-1.0137e-18,  1.5649e-08, -6.3783e-19,  5.0711e-10,  2.0669e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06,  1.1504e-01, -4.4210e-01,
         -4.9752e-01,  3.2580e-01, -9.1061e-01,  3.5500e-02, -7.3294e-02,
          4.6471e-01,  2.8962e-01,  1.4307e-01, -1.0629e+00,  1.7719e-01]],
       device='cuda:0')
torques: [ 1.04535659e-15 -9.12608362e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.23494367e-16  2.26674579e-16 -6.49871253e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.95162031e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.1752, -0.7263, -0.7328,  0.9132, -1.5602,  0.0773, -0.1043,  0.4913,
          0.4185,  0.4710, -1.6359,  0.1081]], device='cuda:0')
Scaled actions :  tensor([[ 0.1752, -0.7263, -0.7328,  0.9132, -1.5602,  0.0773, -0.1043,  0.4913,
          0.4185,  0.4710, -1.6359,  0.1081]], device='cuda:0')
obs :  tensor([[ 8.9486e-02, -8.5412e-02, -1.6906e-01, -2.2916e-03, -1.8038e-03,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.4320e-02,
         -1.2198e-02, -2.0195e-02,  4.1307e-02, -1.0827e-01,  1.1936e-02,
         -2.0564e-02,  1.4345e-03,  3.7563e-04,  2.7424e-02, -1.0586e-01,
          4.6109e-02,  1.9278e-01, -1.0444e-01, -1.7553e-01,  3.5339e-01,
         -9.7399e-01,  5.5143e-02, -1.7353e-01,  1.6194e-02,  2.1754e-02,
          1.9771e-01, -9.5811e-01,  2.6417e-01,  1.7515e-01, -7.2631e-01,
         -7.3283e-01,  9.1316e-01, -1.5602e+00,  7.7267e-02, -1.0430e-01,
          4.9129e-01,  4.1854e-01,  4.7104e-01, -1.6359e+00,  1.0

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.3206, -0.1539,  0.0475,  0.0536, -0.1552, -0.5606,  0.4351,  0.4856,
         -0.3463, -0.4406, -0.2719,  0.2244]], device='cuda:0')
Scaled actions :  tensor([[-0.3206, -0.1539,  0.0475,  0.0536, -0.1552, -0.5606,  0.4351,  0.4856,
         -0.3463, -0.4406, -0.2719,  0.2244]], device='cuda:0')
obs :  tensor([[ 0.1207, -0.1110, -0.4231, -0.0064, -0.0062, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0859, -0.0435, -0.0740,  0.1455, -0.4084,  0.0347, -0.0565,
          0.0106,  0.0090,  0.0835, -0.4060,  0.0795,  0.2338, -0.1947, -0.3460,
          0.6484, -1.9296,  0.0956, -0.1317,  0.0682,  0.0618,  0.3414, -1.9442,
          0.0612, -0.3206, -0.1539,  0.0475,  0.0536, -0.1552, -0.5606,  0.4351,
          0.4856, -0.3463, -0.4406, -0.2719,  0.2244]], device='cuda:0')
torques: [  38.74487581  200.          200.          200.         -200.
   -6.22914178  -54.1624177  -200.         -200.          200.
 -200.          -10.46591188]
データ収集: step 4


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.3251,  0.0090,  0.2177, -0.3081,  0.4389, -0.3806,  0.3882, -0.0168,
         -0.7090, -1.0483,  0.0467,  0.2192]], device='cuda:0')
Scaled actions :  tensor([[-0.3251,  0.0090,  0.2177, -0.3081,  0.4389, -0.3806,  0.3882, -0.0168,
         -0.7090, -1.0483,  0.0467,  0.2192]], device='cuda:0')
obs :  tensor([[-0.1159, -0.0128, -0.1601, -0.0085, -0.0056, -0.9999,  1.0000,  0.0000,
          0.0000,  0.0812, -0.0709, -0.1269,  0.2366, -0.6898, -0.0545, -0.0329,
          0.0372,  0.0160,  0.1339, -0.6865,  0.1255, -0.2284, -0.0919, -0.2025,
          0.3064, -0.9814, -0.8877,  0.3213,  0.1904,  0.0067,  0.1827, -0.9610,
          0.2157, -0.3251,  0.0090,  0.2177, -0.3081,  0.4389, -0.3806,  0.3882,
         -0.0168, -0.7090, -1.0483,  0.0467,  0.2192]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.          200.
  -18.27978941 -200.          -68.23440428  200.         -200.
  200.         -200.        ]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.4232,  0.0098,  0.5551, -0.5072,  0.3615,  1.2531, -0.8877, -1.1071,
         -0.0996, -0.7993, -0.8380, -0.1290]], device='cuda:0')
Scaled actions :  tensor([[ 0.4232,  0.0098,  0.5551, -0.5072,  0.3615,  1.2531, -0.8877, -1.1071,
         -0.0996, -0.7993, -0.8380, -0.1290]], device='cuda:0')
obs :  tensor([[-1.2658e-01,  1.0807e-02,  1.2117e-01, -8.6165e-03, -7.0305e-04,
         -9.9996e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.6295e-02,
         -7.5799e-02, -1.5158e-01,  2.6546e-01, -7.8261e-01, -1.7243e-01,
          7.8029e-02,  6.7252e-02,  8.8052e-03,  1.5362e-01, -7.7116e-01,
          1.5942e-01, -6.3538e-01,  2.3475e-02, -5.5588e-02,  1.6382e-02,
         -4.0967e-02, -4.6137e-01,  6.5644e-01,  1.1583e-01, -7.4554e-02,
          3.2494e-02,  1.6636e-02,  1.3690e-01,  4.2321e-01,  9.7923e-03,
          5.5512e-01, -5.0725e-01,  3.6147e-01,  1.2531e+00, -8.8766e-01,
         -1.1071e+00, -9.9603e-02, -7.9933e-01, -8.3805e-01, -1.2

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 1.3297,  0.1131, -0.5103, -0.3110, -0.1126,  0.6628, -0.8844,  0.1048,
         -0.0732, -0.1508, -0.5321,  0.1637]], device='cuda:0')
Scaled actions :  tensor([[ 1.3297,  0.1131, -0.5103, -0.3110, -0.1126,  0.6628, -0.8844,  0.1048,
         -0.0732, -0.1508, -0.5321,  0.1637]], device='cuda:0')
obs :  tensor([[-0.1704,  0.0340,  0.3252, -0.0079,  0.0052, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0953, -0.0722, -0.1392,  0.2385, -0.6870, -0.1583,  0.1579,
          0.0856, -0.0032,  0.1414, -0.7916,  0.0964, -0.1991,  0.0148,  0.1559,
         -0.2616,  0.9034,  0.5062,  0.1941,  0.0700, -0.0494, -0.1388, -0.0998,
         -0.4798,  1.3297,  0.1131, -0.5103, -0.3110, -0.1126,  0.6628, -0.8844,
          0.1048, -0.0732, -0.1508, -0.5321,  0.1637]], device='cuda:0')
torques: [-200.         -200.         -142.7905913  -200.            7.06562866
   21.24976917  200.          149.39472129  200.         -200.
  200.          200.        ]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.0238, -0.8327, -0.7453,  0.6118, -1.5374, -0.1920, -0.2387, -0.3615,
          0.7910,  0.8766, -0.4859,  0.1477]], device='cuda:0')
Scaled actions :  tensor([[-0.0238, -0.8327, -0.7453,  0.6118, -1.5374, -0.1920, -0.2387, -0.3615,
          0.7910,  0.8766, -0.4859,  0.1477]], device='cuda:0')
obs :  tensor([[-0.3602,  0.4692,  0.1499,  0.0042,  0.0173, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0889, -0.0582, -0.1204,  0.1669, -0.4889,  0.0270,  0.1491,
          0.1046, -0.0399,  0.1470, -0.7405,  0.0661,  0.2201,  0.0971,  0.0602,
         -0.4390,  0.8295,  1.2569, -0.2106,  0.0729, -0.2487,  0.0898,  0.3907,
          0.1132, -0.0238, -0.8327, -0.7453,  0.6118, -1.5374, -0.1920, -0.2387,
         -0.3615,  0.7910,  0.8766, -0.4859,  0.1477]], device='cuda:0')
torques: [-200.         -172.68401035  200.         -200.           45.00209689
  119.81054776  200.          200.         -200.         -200.
  -81.29756597  125.02784764]
データ収集:

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.0527, -0.4818,  0.4525,  0.5783, -0.3323, -0.5372,  0.5818,  0.9589,
          0.7988, -0.2216, -0.0930,  0.2708]], device='cuda:0')
Scaled actions :  tensor([[ 0.0527, -0.4818,  0.4525,  0.5783, -0.3323, -0.5372,  0.5818,  0.9589,
          0.7988, -0.2216, -0.0930,  0.2708]], device='cuda:0')
obs :  tensor([[ 0.3968,  0.3163,  0.2335,  0.0202,  0.0147, -0.9997,  1.0000,  0.0000,
          0.0000, -0.0475, -0.0724, -0.1399,  0.1499, -0.4220,  0.1524,  0.0543,
          0.0916, -0.0765,  0.1780, -0.6485,  0.1105,  0.1501, -0.1975, -0.2340,
          0.2414, -0.0660,  0.0912, -0.5745, -0.1825, -0.1391,  0.2063,  0.4252,
          0.1963,  0.0527, -0.4818,  0.4525,  0.5783, -0.3323, -0.5372,  0.5818,
          0.9589,  0.7988, -0.2216, -0.0930,  0.2708]], device='cuda:0')
torques: [ 200. -200.  200.  200. -200. -200. -200. -200. -200.  200. -200. -200.]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.1586,  0.3467,  0.0114, -0.5504, -0.7606,  0.1235,  1.2494,  0.5645,
         -0.1183, -0.8953, -1.0416, -0.3749]], device='cuda:0')
Scaled actions :  tensor([[-0.1586,  0.3467,  0.0114, -0.5504, -0.7606,  0.1235,  1.2494,  0.5645,
         -0.1183, -0.8953, -1.0416, -0.3749]], device='cuda:0')
obs :  tensor([[ 0.4556, -0.3477, -0.0122,  0.0180, -0.0023, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0031, -0.1215, -0.1624,  0.2112, -0.3960,  0.0654, -0.0126,
          0.0586, -0.0687,  0.1898, -0.4967,  0.1661,  0.1944, -0.2735, -0.0141,
          0.3571,  0.1360, -0.8673, -0.1458, -0.1569,  0.1877, -0.0626,  0.8653,
          0.2279, -0.1586,  0.3467,  0.0114, -0.5504, -0.7606,  0.1235,  1.2494,
          0.5645, -0.1183, -0.8953, -1.0416, -0.3749]], device='cuda:0')
torques: [ 200.          200.          200.         -200.          -48.61179166
  -18.96334172  -81.79798825 -200.          200.          200.
   -8.32841864 -200.        ]
データ収集:

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.3122,  0.0356, -0.8783, -0.5400, -0.4091,  0.5800, -0.5264,  0.3598,
         -0.7627, -0.1324, -1.1261, -0.0675]], device='cuda:0')
Scaled actions :  tensor([[-0.3122,  0.0356, -0.8783, -0.5400, -0.4091,  0.5800, -0.5264,  0.3598,
         -0.7627, -0.1324, -1.1261, -0.0675]], device='cuda:0')
obs :  tensor([[-0.1538, -0.3458,  0.1555,  0.0038, -0.0071, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0118, -0.1424, -0.1425,  0.2464, -0.4731, -0.0117,  0.0047,
          0.0580, -0.0415,  0.1780, -0.4330,  0.1019, -0.2194,  0.0278,  0.1804,
          0.0317, -0.6542,  0.0345,  0.2693,  0.1347,  0.0720, -0.0186, -0.1224,
         -0.7748, -0.3122,  0.0356, -0.8783, -0.5400, -0.4091,  0.5800, -0.5264,
          0.3598, -0.7627, -0.1324, -1.1261, -0.0675]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.         -200.
 -200.         -155.92794404  200.          170.9771056  -200.
  200.          200.        ]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.4348, -1.0682, -0.8882,  0.0665, -0.0481, -0.0974, -0.5367, -0.8292,
          0.5188,  0.3467, -0.6959,  0.6837]], device='cuda:0')
Scaled actions :  tensor([[ 0.4348, -1.0682, -0.8882,  0.0665, -0.0481, -0.0974, -0.5367, -0.8292,
          0.5188,  0.3467, -0.6959,  0.6837]], device='cuda:0')
obs :  tensor([[-6.4609e-01,  3.4275e-01,  6.4564e-01,  4.9002e-03,  9.8995e-03,
         -9.9994e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.0510e-01,
         -1.1019e-01, -1.3065e-01,  2.3448e-01, -5.1090e-01,  9.1490e-02,
          4.6644e-04,  1.1172e-01, -5.0232e-02,  1.7886e-01, -5.4852e-01,
         -6.0622e-02, -6.2855e-01,  2.7303e-01, -2.9947e-02, -1.3336e-01,
          1.8498e-01,  7.8765e-01, -2.4335e-01,  3.7069e-01, -1.3394e-01,
          1.1262e-02, -9.6033e-01, -8.1786e-01,  4.3477e-01, -1.0682e+00,
         -8.8823e-01,  6.6543e-02, -4.8097e-02, -9.7423e-02, -5.3669e-01,
         -8.2915e-01,  5.1884e-01,  3.4673e-01, -6.9585e-01,  6.

In [29]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.811, Scaled action max=0.811
Step 1/100, Total steps: 512
steps: 512
actions : tensor([[-0.7006,  0.5538, -1.0895, -0.0160, -0.7179, -0.0958, -0.6343,  0.1860,
         -0.9717, -0.0158,  0.3825,  0.8108]], device='cuda:0')
target_dof_pos: tensor([[ 0.0782, -0.9800, -1.3533,  1.5126, -1.2600, -1.1874,  0.7159, -0.6809,
         -1.4119,  1.3366, -0.8223, -0.4796]], device='cuda:0')
Step 1: Original action max=1.143, Scaled action max=1.143
Step 2: Original action max=1.015, Scaled action max=1.015
Step 21/100, Total steps: 532
steps: 532
actions : tensor([[-0.1491,  0.4164,  0.3187, -0.0266, -1.3450,  1.0797,  0.4323,  0.2675,
         -0.0030,  0.7173, -0.5821, -0.0203]], device='cuda:0')
target_dof_pos: tensor([[ 0.7709, -0.1447, -0.4328,  1.3114, -1.3302,  0.4889,  0.6230,  0.2415,
         -0.2429,  1.5995, -0.7186, -0.2689]], device='cuda:0')
Step 41/100, Total steps: 552
steps: 552
actions : tensor([[ 0.2881, -1.0909,  0.2238, -0.2213, -1.0112, -0.14

In [45]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [30]:
env.sim.stop()

In [43]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
